#### Objetivo

- Esse notebook consiste em concatenar os dados dos eventos de todas as partidas das competições entre todas as suas temporadas. 

    A ideia é de analisar principalmente a disponibilidade dos dados de tracking e entender quais partidas fariam mais sentido de serem utilizadas.

In [1]:
import pandas as pd
from pathlib import Path
import os
from pyspark.sql import SparkSession

pd.set_option('display.max_columns', None)

In [2]:
competitions = [competition for competition in os.listdir(str(Path().resolve().parent.parent / "data" / "events"))]
competitions

['1', '42']

In [3]:
competitions_seasons = {
    competitions[id]: [season for season in os.listdir(str(Path().resolve().parent.parent / "data" / "events" / competitions[id]))]
    for id in range(len(competitions))
}
competitions_seasons

{'1': ['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025'],
 '42': ['2023', '2024', '2025']}

In [5]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [6]:
def get_competition_seasons_parquet_file_paths(competition, seasons):

    competition_seasons_parquet_file_paths = []

    for season in seasons:

        events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / competition / season)

        competition_seasons_parquet_file_paths.extend(get_season_events_parquet_file_paths(events_competition_season_folder_path))
    
    return competition_seasons_parquet_file_paths

In [7]:
competitions_seasons_events_parquet_file_paths = []

for competition, seasons in competitions_seasons.items():

    competitions_seasons_events_parquet_file_paths.extend(get_competition_seasons_parquet_file_paths(competition, seasons))

competitions_seasons_events_parquet_file_paths

['C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2020-2021\\events_1_2020-2021_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2020-2021\\events_1_2020-2021_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2021-2022\\events_1_2021-2022_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2021-2022\\events_1_2021-2022_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2022-2023\\events_1_2022-2023_part1.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-prediction\\data\\events\\1\\2022-2023\\events_1_2022-2023_part2.parquet',
 'C:\\Users\\Matheus\\Documents\\Mestrado\\Dissertação\\defensive-performance-pred

In [8]:
spark = SparkSession.builder.master("local[*]").appName("events").getOrCreate()

df = spark.read.parquet(*competitions_seasons_events_parquet_file_paths)

df.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+---------+-----------+-------+------------+
|                  id|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|             details|         homePlayers|         awayPlayers|               balls|player.id|player.name|team.id|   team.name|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+--------------------+--------------------+--------------------+--------------------+---------+-----------+-------+------------+
|4a60f04fd9c90fe77...|            1|   152|2020-2021|     1|       First half|FIRSTKICKOFF| First half kick off|             0|         

In [9]:
df.count()

7580243